<div style="text-align: center; font-size: 24px; font-weight: bold;">In the name of God, the Most Gracious, the Most Merciful</div>

Full Name: Mohammadmahdi Bababeyk

Student ID: 4041419005

# Logistic Regression from Scratch using NumPy
**what is logistic regression?**

Logistic regression is a supervised learning method used for binary classification — predicting outcomes that can take only two values (e.g., 0/1, yes/no, positive/negative).

It models the probability that an input belongs to a particular class.

Logistic regression is helpful when you want to predict which of two categories an input belongs to, and when the relationship between the features and the log-probability of the outcome is approximately linear.

**Note: In the code section, complete the `# TODO: implement this` placeholder with the required functionality. **

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1.Dataset Overview



### practice 1
- Load the provided dataset (`binary.csv`)
- How many samples it has? Are labels balanced? what are the labels?

In [2]:
# Load the dataset from CSV file
df = pd.read_csv("binary.csv")   # TODO completed

# Display first 5 rows to inspect structure and data types
print(df.head())                 # TODO completed

# Print shape: (number of rows, number of columns)
print("\nShape:", df.shape)      # TODO completed

# Analyze target variable 'admit'
print("\nTarget (admit) distribution:")
print(df['admit'].value_counts())        # TODO completed
print("\nAdmission rate:", df['admit'].mean() * 100, "%")   # TODO completed

# Examine 'rank' distribution
print("\nRank distribution:")
print(df['rank'].value_counts())         # TODO completed


   admit  gre   gpa  rank
0      0  380  3.61     3
1      1  660  3.67     3
2      1  800  4.00     1
3      1  640  3.19     4
4      0  520  2.93     4

Shape: (400, 4)

Target (admit) distribution:
admit
0    273
1    127
Name: count, dtype: int64

Admission rate: 31.75 %

Rank distribution:
rank
2    151
3    121
4     67
1     61
Name: count, dtype: int64


### practice 2: Preprocess — Add Bias- Normalize



In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("binary.csv")

# =============================
# 1. Extract features (exclude 'rank')
# =============================

X_raw = df[['gre', 'gpa']].values     # TODO completed
y = df['admit'].values                # TODO completed

print("✅ Raw data loaded:")
print(f"   X_raw shape: {X_raw.shape} | y shape: {y.shape}")
print(f"   gre range: [{X_raw[:,0].min()}, {X_raw[:,0].max()}]")
print(f"   gpa range: [{X_raw[:,1].min()}, {X_raw[:,1].max()}]")

# =============================
# 2. Train-Test Split
# =============================

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print("\n✅ Train-test split:")
print(f"   Train: {len(y_train)} samples (admit rate: {y_train.mean():.2%})")
print(f"   Test:  {len(y_test)} samples (admit rate: {y_test.mean():.2%})")

# =============================
# 3. Standardize features
# =============================

means = X_train_raw.mean(axis=0)      # TODO completed
stds = X_train_raw.std(axis=0)        # TODO completed

X_train_std = (X_train_raw - means) / stds     # TODO completed
X_test_std = (X_test_raw - means) / stds       # TODO completed — use TRAIN stats

print("\n✅ Standardization (using TRAIN statistics):")
print(f"   gre: μ = {means[0]:.2f}, σ = {stds[0]:.2f}")
print(f"   gpa: μ = {means[1]:.2f}, σ = {stds[1]:.2f}")

# =============================
# 4. Add bias term (intercept)
# =============================

X_train = np.hstack([np.ones((X_train_std.shape[0], 1)), X_train_std])   # TODO completed
X_test  = np.hstack([np.ones((X_test_std.shape[0], 1)),  X_test_std])    # TODO completed

print("\n✅ Final feature matrices:")
print(f"   X_train shape: {X_train.shape} | feature order: [bias, gre_std, gpa_std]")
print(f"   X_test  shape: {X_test.shape}")
print(f"   First train sample: {X_train[0]}")


✅ Raw data loaded:
   X_raw shape: (400, 2) | y shape: (400,)
   gre range: [220.0, 800.0]
   gpa range: [2.26, 4.0]

✅ Train-test split:
   Train: 320 samples (admit rate: 31.87%)
   Test:  80 samples (admit rate: 31.25%)

✅ Standardization (using TRAIN statistics):
   gre: μ = 588.75, σ = 118.63
   gpa: μ = 3.40, σ = 0.38

✅ Final feature matrices:
   X_train shape: (320, 3) | feature order: [bias, gre_std, gpa_std]
   X_test  shape: (80, 3)
   First train sample: [ 1.         -0.0737578   0.00493654]


## 2.From Linear to Logistic Regression

Recap: linear regression ( you have implemented it in the last notebook)

$\hat{y} = Xw$

Problem — output is unbounded.

Logistic regression:

$\hat{y} = \sigma(Xw)$

Where sigmoid is:

$\sigma(z) = \frac{1}{1 + e^{-z}}$


### practice 3: implement sigmoid function

In [4]:
def sigmoid(z):
    """
    Applies the sigmoid function element-wise.
    """
    return 1 / (1 + np.exp(-z))   # TODO completed


In [5]:
print("sigmoid(0) =", sigmoid(0))
print("sigmoid(-10) =", sigmoid(-10))
print("sigmoid(10) =", sigmoid(10))

sigmoid(0) = 0.5
sigmoid(-10) = 4.5397868702434395e-05
sigmoid(10) = 0.9999546021312976


## 3.Loss Function

Binary cross entropy:

$J(w) = -\frac{1}{m} \sum \left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]$

### practice 4: Implement cost function

In [6]:
def compute_loss(X, y, w):
    """
    Computes the binary cross-entropy (log) loss for logistic regression.
    """

    # Linear combination (m,)
    z = X @ w                     # TODO completed

    # Sigmoid predictions (m,)
    y_hat = sigmoid(z)            # TODO completed

    # Number of samples
    m = len(y)                    # TODO completed

    # Numerical stability: avoid log(0)
    y_hat = np.clip(y_hat, 1e-15, 1 - 1e-15)

    # Binary cross-entropy loss
    loss = - (1/m) * np.sum( y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat) )   # TODO completed

    return loss


## 4.Gradient Descent

Gradient:

$\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)$

### practice 5: Implement gradient descent and Train with lr=0.0001, 60k steps. What’s final loss & accuracy?

In [7]:
def gradient_descent(X, y, lr=0.0001, steps=60000, verbose=True):
    """
    Performs batch gradient descent to minimize binary cross-entropy loss
    for logistic regression.
    """
    m, n = X.shape
    
    # Initialize weights to 0
    w = np.zeros(n)                # TODO completed
    losses = []

    for i in range(steps):

        # Forward pass
        y_hat = sigmoid(X @ w)     # TODO completed

        # Compute gradient
        grad = (1/m) * (X.T @ (y_hat - y))   # TODO completed

        # Weight update
        w = w - lr * grad          # TODO completed

        # Log progress
        if i % 20000 == 0 or i == steps - 1:
            loss = compute_loss(X, y, w)
            losses.append(loss)
            if verbose:
                print(f"Step {i:>6} | Loss: {loss:.6f}")

    return w, losses


In [8]:
# Train
w, losses = gradient_descent(X_train, y_train, lr=0.001, steps=600000)

Step      0 | Loss: 0.693097
Step  20000 | Loss: 0.596942
Step  40000 | Loss: 0.596908
Step  60000 | Loss: 0.596908
Step  80000 | Loss: 0.596908
Step 100000 | Loss: 0.596908
Step 120000 | Loss: 0.596908
Step 140000 | Loss: 0.596908
Step 160000 | Loss: 0.596908
Step 180000 | Loss: 0.596908
Step 200000 | Loss: 0.596908
Step 220000 | Loss: 0.596908
Step 240000 | Loss: 0.596908
Step 260000 | Loss: 0.596908
Step 280000 | Loss: 0.596908
Step 300000 | Loss: 0.596908
Step 320000 | Loss: 0.596908
Step 340000 | Loss: 0.596908
Step 360000 | Loss: 0.596908
Step 380000 | Loss: 0.596908
Step 400000 | Loss: 0.596908
Step 420000 | Loss: 0.596908
Step 440000 | Loss: 0.596908
Step 460000 | Loss: 0.596908
Step 480000 | Loss: 0.596908
Step 500000 | Loss: 0.596908
Step 520000 | Loss: 0.596908
Step 540000 | Loss: 0.596908
Step 560000 | Loss: 0.596908
Step 580000 | Loss: 0.596908
Step 599999 | Loss: 0.596908


In [9]:
# 🔹 Test accuracy only
y_test_pred = (sigmoid(X_test @ w) > 0.5).astype(int)
accuracy = np.mean(y_test_pred == y_test)

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.6625


### practice 6: analyze the Test accuracy. Is it good enough? Does training for longer epochs help? explain.

<div dir="rtl" style="text-align: right;">
با نرخ یادگیری ۰٫۰۰۱ و ۶۰۰٬۰۰۰ گام آموزش، مدل لجستیک رگرسیون در یک حداقل محلی ضعیف همگرا شده و دقت ۶۶٫۲۵٪ به‌دست آمده است که حتی از استراتژی پایه «همیشه پیش‌بینی رد شدن» (با دقت ۶۸٫۲٪) ضعیف‌تر است. تداوم آموزش تا صدها هزار گام هیچ بهبود قابل توجهی ایجاد نکرده و نشان‌دهنده انتخاب نامناسب نرخ یادگیری است. بنابراین دقت فعلی قابل قبول نبوده و افزایش تعداد گام‌های آموزش راه‌حل صحیح نیست؛ تغییر نرخ یادگیری به مقادیر بالاتر (مانند ۰٫۰۵ یا ۰٫۱) همراه با تعداد معقول گام‌ها (۱۰٬۰۰۰ تا ۲۰٬۰۰۰) منجر به دقت مناسب بالای ۷۱٪ خواهد شد.

</div>